In [15]:
%run /Users/manojrammopati/Downloads/bupa_insurance_project/01_Bronze_Layer/Notebooks/_01_bronze_container_connect.ipynb

Preconnectors ran successfully and connected to 'clientdatastorage' storage accounts
 'policy_df, claim_df, member_df, provider_df' are ready to use from container 'raw data'(meridian architecture of bronze) 


In [13]:
import sys
import importlib
import utils_silver
from utils_silver import *

# Add the Silver layer directory to sys.path and import the utils module

silver_dir = "/Users/manojrammopati/Downloads/bupa_insurance_project/02_Silver_Layer"
if silver_dir not in sys.path:
    sys.path.insert(0, silver_dir)

# Now import (and reload to pick up edits)
importlib.reload(utils_silver)


paths_bronze, paths_silver = utils_silver.build_paths(storage_account1)
print(sorted(paths_silver.keys()))
# Optionally bring helper names into the notebook namespace (avoid if you prefer module namespace)

print("Imported utils_silver from", utils_silver.__file__)

✅ utils_silver.py loaded
['_metrics', '_quarantine', '_ref_dim_channel', '_ref_dim_product_line', '_reference', 'claims', 'members', 'policies', 'providers']
Imported utils_silver from /Users/manojrammopati/Downloads/bupa_insurance_project/02_Silver_Layer/utils_silver.py


# 🔹 Cell 1 — Imports & environment / paths

In [4]:
# --- Cell 1: Setup & Imports ---

from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql import SparkSession

from utils_silver import (
    build_paths,
    enforce_schema,
    check_duplicates,
    data_quality_summary,
    normalize_categories,
    dq_expect,
    dq_left_anti_ref,
    fk_filter,
    write_metric,
)

# Storage account for this project
STORAGE_ACCOUNT = "clientdatastorage"

# Build bronze/silver paths using your shared utils
paths_bronze, paths_silver = build_paths(STORAGE_ACCOUNT)

print("BRONZE PATHS:", paths_bronze)
print("SILVER PATHS:", paths_silver)

# Logical DB name (Hive/Delta metastore)
DB_SILVER = "bupa_silver"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_SILVER}")
spark.sql(f"USE {DB_SILVER}")

print("✅ Environment ready (Members Silver)")


BRONZE PATHS: {'policies': 'abfss://rawdata@clientdatastorage.dfs.core.windows.net/policies', 'members': 'abfss://rawdata@clientdatastorage.dfs.core.windows.net/members', 'claims': 'abfss://rawdata@clientdatastorage.dfs.core.windows.net/claims', 'providers': 'abfss://rawdata@clientdatastorage.dfs.core.windows.net/providers'}
SILVER PATHS: {'policies': 'abfss://silverdata@clientdatastorage.dfs.core.windows.net/policies', 'members': 'abfss://silverdata@clientdatastorage.dfs.core.windows.net/members', 'claims': 'abfss://silverdata@clientdatastorage.dfs.core.windows.net/claims', 'providers': 'abfss://silverdata@clientdatastorage.dfs.core.windows.net/providers', '_quarantine': 'abfss://silverdata@clientdatastorage.dfs.core.windows.net/_quarantine', '_metrics': 'abfss://silverdata@clientdatastorage.dfs.core.windows.net/_metrics', '_reference': 'abfss://silverdata@clientdatastorage.dfs.core.windows.net/reference', '_ref_dim_channel': 'abfss://silverdata@clientdatastorage.dfs.core.windows.net/

# 🔹 Cell 2 — Read Bronze Members & enforce schema

In [5]:
# --- Cell 2: Read Bronze Members & enforce base schema ---

BRONZE_MEMBERS_PATH = paths_bronze["members"]
print("Reading Bronze Members from:", BRONZE_MEMBERS_PATH)

members_bz = spark.read.format("delta").load(BRONZE_MEMBERS_PATH)

print("Bronze Members schema:")
members_bz.printSchema()
print("Bronze Members row count:", members_bz.count())

# Target schema for Silver Members
members_schema = StructType([
    StructField("Member_ID",         StringType()),
    StructField("Policy_ID",         StringType()),
    StructField("Age",               IntegerType()),
    StructField("Gender",            StringType()),
    StructField("BMI",               DoubleType()),
    StructField("Smoker",            StringType()),
    StructField("Chronic_Disease",   StringType()),
    StructField("Employment_Status", StringType()),
    StructField("Region",            StringType())
])

members = enforce_schema(members_bz, members_schema)

print("After enforce_schema (Members):")
members.printSchema()
print("Row count (should match Bronze):", members.count())


Reading Bronze Members from: abfss://rawdata@clientdatastorage.dfs.core.windows.net/members
Bronze Members schema:
root
 |-- Member_ID: string (nullable = true)
 |-- Policy_ID: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- BMI: double (nullable = true)
 |-- Smoker: string (nullable = true)
 |-- Chronic_Disease: string (nullable = true)
 |-- Employment_Status: string (nullable = true)
 |-- Region: string (nullable = true)



25/11/28 13:46:19 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Bronze Members row count: 381109
After enforce_schema (Members):
root
 |-- Member_ID: string (nullable = true)
 |-- Policy_ID: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- BMI: double (nullable = true)
 |-- Smoker: string (nullable = true)
 |-- Chronic_Disease: string (nullable = true)
 |-- Employment_Status: string (nullable = true)
 |-- Region: string (nullable = true)

Row count (should match Bronze): 381109


# 🔹 Cell 3 — Basic DQ summary & duplicate check

In [6]:
# --- Cell 3: Basic DQ summary & duplicate check (Members) ---

# Quick DQ summary (null %, schema, row count)
data_quality_summary(members, "members_bronze_conformed")

# Deduplicate per Member_ID (latest wins if duplicates exist)
members = check_duplicates(members, ["Member_ID"])

print("Row count after dedupe on Member_ID:", members.count())



===== Data Quality Report for members_bronze_conformed =====
root
 |-- Member_ID: string (nullable = true)
 |-- Policy_ID: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- BMI: double (nullable = true)
 |-- Smoker: string (nullable = true)
 |-- Chronic_Disease: string (nullable = true)
 |-- Employment_Status: string (nullable = true)
 |-- Region: string (nullable = true)

Row Count: 381109
Null Percentage:


+---------+---------+---+------+---+------+---------------+-----------------+------+
|Member_ID|Policy_ID|Age|Gender|BMI|Smoker|Chronic_Disease|Employment_Status|Region|
+---------+---------+---+------+---+------+---------------+-----------------+------+
|0.0      |0.0      |0.0|0.0   |0.0|0.0   |0.0            |0.0              |0.0   |
+---------+---------+---+------+---+------+---------------+-----------------+------+



✅ No duplicate keys found.


Row count after dedupe on Member_ID: 381109


# 🔹 Cell 4 — Key checks & hard DQ (null PKs)

In [7]:
# --- Cell 4: Key checks (Member_ID, Policy_ID must exist) ---

# Quarantine rows where keys are missing, but don't fail yet
members_bad_keys = members.filter(
    F.col("Member_ID").isNull() | F.col("Policy_ID").isNull()
)
members_ok = members.filter(
    F.col("Member_ID").isNotNull() & F.col("Policy_ID").isNotNull()
)

if members_bad_keys.count() > 0:
    from utils_silver import quarantine  # already imported, but safe
    quarantine(members_bad_keys, "null_key_member_or_policy", "members", paths_silver)
    print("🚧 Quarantined rows with null Member_ID or Policy_ID:", members_bad_keys.count())
else:
    print("✅ No null Member_ID / Policy_ID rows in Members.")

print("Members surviving key check:", members_ok.count())


✅ No null Member_ID / Policy_ID rows in Members.


Members surviving key check: 381109


# 🔹 Cell 5 — Normalisation & soft DQ flags (age, BMI)

In [8]:
# --- Cell 5: Normalise categories & add DQ flags (Members) ---

# 1) Normalise Gender/Smoker using shared utility
members_ok = normalize_categories(members_ok)

# 2) Clean Employment_Status for consistency (title case)
members_ok = members_ok.withColumn(
    "Employment_Status",
    F.initcap(F.trim(F.col("Employment_Status")))
)

# 3) DQ flags
members_ok = (
    members_ok
    .withColumn(
        "dq_age_valid",
        F.when((F.col("Age").isNull()) | ((F.col("Age") >= 0) & (F.col("Age") <= 110)), 1).otherwise(0)
    )
    .withColumn(
        "dq_bmi_valid",
        F.when(
            (F.col("BMI").isNull()) |
            ((F.col("BMI") >= 10.0) & (F.col("BMI") <= 60.0)),
            1
        ).otherwise(0)
    )
)

print("Preview after normalization & DQ flags:")
members_ok.show(5, truncate=False)


Preview after normalization & DQ flags:


+------------+----------+---+------+------------------+------+---------------+-----------------+------+------------+------------+
|Member_ID   |Policy_ID |Age|Gender|BMI               |Smoker|Chronic_Disease|Employment_Status|Region|dq_age_valid|dq_bmi_valid|
+------------+----------+---+------+------------------+------+---------------+-----------------+------+------------+------------+
|MEM_00000008|P_00000008|56 |F     |23.140940890221643|Y     |Asthma         |Student          |280   |1           |1           |
|MEM_00000009|P_00000009|24 |F     |26.143483864989406|N     |None           |Employed         |30    |1           |1           |
|MEM_00000018|P_00000018|25 |F     |32.90835412689146 |N     |None           |Retired          |350   |1           |1           |
|MEM_00000023|P_00000023|23 |M     |24.171598788575466|N     |Diabetes       |Employed         |500   |1           |1           |
|MEM_00000039|P_00000039|45 |M     |25.203535452103754|Y     |None           |Employed    

# 🔹 Cell 6 — FK validation vs Policies & Business features

In [9]:
# --- Cell 6: FK validation vs Policies & feature engineering ---

# 1) Load Silver Policies to validate Policy_ID
SILVER_POLICIES_PATH = paths_silver["policies"]

try:
    policies_silver = spark.table(f"{DB_SILVER}.policies")
    print("✅ Loaded Silver Policies from table:", f"{DB_SILVER}.policies")
except Exception:
    policies_silver = spark.read.format("delta").load(SILVER_POLICIES_PATH)
    print("✅ Loaded Silver Policies from path:", SILVER_POLICIES_PATH)

# 2) FK filter: Members must link to a valid Policy_ID
members_fk_ok = fk_filter(
    members_ok,
    fk_col="Policy_ID",
    ref_df=policies_silver,
    ref_key="Policy_ID",
    table_label="members",
    reason="fk_policy_not_found",
    paths_silver=paths_silver
)

print("Row count after FK filter vs Policies:", members_fk_ok.count())

# 3) Feature engineering: Age band, BMI category, Risk segment

def age_band_col(col):
    return (
        F.when(col < 25, "<25")
         .when(col.between(25, 40), "25-40")
         .when(col.between(41, 60), "41-60")
         .otherwise("60+")
    )

def bmi_category_col(col):
    return (
        F.when(col.isNull(), None)
         .when(col < 18.5, "Underweight")
         .when(col < 25, "Normal")
         .when(col < 30, "Overweight")
         .otherwise("Obese")
    )

members_enriched = (
    members_fk_ok
    .withColumn("Age_Band", age_band_col(F.col("Age")))
    .withColumn("BMI_Category", bmi_category_col(F.col("BMI")))
    .withColumn(
        "Risk_Segment",
        F.when(
            (F.col("Chronic_Disease").isin("Hypertension", "Diabetes")) |
            (F.col("Smoker") == "Y") |
            (F.col("BMI") >= 30),
            "High"
        )
        .when(
            (F.col("Chronic_Disease") == "Asthma") |
            ((F.col("BMI") >= 25) & (F.col("BMI") < 30)),
            "Medium"
        )
        .otherwise("Low")
    )
)

print("Preview enriched Members:")
members_enriched.show(5, truncate=False)


✅ Loaded Silver Policies from path: abfss://silverdata@clientdatastorage.dfs.core.windows.net/policies


Row count after FK filter vs Policies: 381109
Preview enriched Members:


+------------+----------+---+------+------------------+------+---------------+-----------------+------+------------+------------+--------+------------+------------+
|Member_ID   |Policy_ID |Age|Gender|BMI               |Smoker|Chronic_Disease|Employment_Status|Region|dq_age_valid|dq_bmi_valid|Age_Band|BMI_Category|Risk_Segment|
+------------+----------+---+------+------------------+------+---------------+-----------------+------+------------+------------+--------+------------+------------+
|MEM_00000177|P_00000177|25 |M     |32.704279210209876|N     |Diabetes       |Employed         |280   |1           |1           |25-40   |Obese       |High        |
|MEM_00000349|P_00000349|31 |F     |22.455539400786847|Y     |Asthma         |Employed         |280   |1           |1           |25-40   |Normal      |High        |
|MEM_00000465|P_00000465|24 |M     |25.924970590479838|Y     |Asthma         |Retired          |20    |1           |1           |<25     |Overweight  |High        |
|MEM_00000

# 🔹 Cell 7 — DQ rules with dq_expect / dq_left_anti_ref

In [10]:
# --- Cell 7: DQ rules using dq_expect & dictionary checks ---

from utils_silver import dq_expect, dq_left_anti_ref

# Normalize flags and expectations:

# 1) Critical expectations (hard-error)
dq_expect(
    members_enriched,
    name="key_not_null",
    expr="Member_ID IS NOT NULL AND Policy_ID IS NOT NULL",
    severity="error",
    table_label="members",
    paths_silver=paths_silver
)

dq_expect(
    members_enriched,
    name="age_bounds",
    expr="Age IS NULL OR (Age >= 0 AND Age <= 110)",
    severity="error",
    table_label="members",
    paths_silver=paths_silver
)

dq_expect(
    members_enriched,
    name="bmi_bounds",
    expr="BMI IS NULL OR (BMI >= 10 AND BMI <= 60)",
    severity="error",
    table_label="members",
    paths_silver=paths_silver
)

# 2) Dictionary-style validations (soft-error / error)
#    Build small reference DataFrames for categories

dim_smoker = spark.createDataFrame(
    [("Y",), ("N",)],
    "Smoker STRING"
)

dim_gender = spark.createDataFrame(
    [("M",), ("F",)],
    "Gender STRING"
)

# Validate Smoker in {'Y','N'} where not null
dq_left_anti_ref(
    members_enriched.filter("Smoker IS NOT NULL"),
    dim_smoker,
    key_col="Smoker",
    ref_col="Smoker",
    name="smoker_in_dictionary",
    severity="error",
    table_label="members",
    paths_silver=paths_silver
)

# Validate Gender in {'M','F'} where not null
dq_left_anti_ref(
    members_enriched.filter("Gender IS NOT NULL"),
    dim_gender,
    key_col="Gender",
    ref_col="Gender",
    name="gender_in_dictionary",
    severity="error",
    table_label="members",
    paths_silver=paths_silver
)

print("✅ DQ checks complete for Members.")


✅ DQ PASS [members] key_not_null


✅ DQ PASS [members] age_bounds


✅ DQ PASS [members] bmi_bounds


✅ DQ PASS [members] smoker_in_dictionary


✅ DQ PASS [members] gender_in_dictionary
✅ DQ checks complete for Members.


# 🔹 Cell 8 — Write Silver Members (Delta + table) & OPTIMIZE

In [11]:
# --- Cell 8: Write Silver Members as Delta + register table ---

SILVER_MEMBERS_PATH = paths_silver["members"]
full_table = f"{DB_SILVER}.members"

print("Writing Silver Members to path:", SILVER_MEMBERS_PATH)

# Coalesce to reduce small files
members_silver_out = members_enriched.coalesce(4)

(
    members_silver_out
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_MEMBERS_PATH)
)

print("✅ Silver Members written to Delta path.")

# Register / replace table on top of that path
spark.sql(f"DROP TABLE IF EXISTS {full_table}")
spark.sql(f"""
CREATE TABLE {full_table}
USING DELTA
LOCATION '{SILVER_MEMBERS_PATH}'
""")

print("✅ Registered Silver Members table:", full_table)

# Optional: OPTIMIZE / ZORDER (best-effort)
def optimize_zorder(table_full_name: str, zcols: list[str]):
    try:
        if zcols:
            cols = ", ".join([f"`{c}`" for c in zcols])
            spark.sql(f"OPTIMIZE {table_full_name} ZORDER BY ({cols})")
        else:
            spark.sql(f"OPTIMIZE {table_full_name}")
        print(f"✅ OPTIMIZE completed for {table_full_name}")
    except Exception as e:
        print(f"[WARN] OPTIMIZE skipped or unavailable: {e}")

optimize_zorder(full_table, ["Policy_ID", "Member_ID"])


Writing Silver Members to path: abfss://silverdata@clientdatastorage.dfs.core.windows.net/members


✅ Silver Members written to Delta path.
✅ Registered Silver Members table: bupa_silver.members


✅ OPTIMIZE completed for bupa_silver.members


# 🔹 Cell 9 — Metrics (observability)

In [16]:
# --- Cell 9: Write metrics for Members Silver ---

from utils_silver import write_metric

members_silver = spark.read.format("delta").load(SILVER_MEMBERS_PATH)

write_metric(
    spark,
    "rowcount_members_silver",
    members_silver.count(),
    "members_silver",
    paths_silver
)

write_metric(
    spark,
    "distinct_member_ids",
    members_silver.select("Member_ID").distinct().count(),
    "members_silver",
    paths_silver
)

metrics_df = spark.read.format("delta").load(paths_silver["_metrics"])
metrics_df.orderBy(F.col("ts").desc()).show(truncate=False)


[METRIC] rowcount_members_silver=381109 ctx=members_silver


[METRIC] distinct_member_ids=381109 ctx=members_silver


+--------------------------+------+---------------+--------------------------+
|metric                    |value |context        |ts                        |
+--------------------------+------+---------------+--------------------------+
|distinct_member_ids       |381109|members_silver |2025-11-28 14:04:23.229343|
|rowcount_members_silver   |381109|members_silver |2025-11-28 14:04:14.937709|
|distinct_policy_ids_silver|381109|policies_silver|2025-11-28 10:36:21.144919|
|rowcount_policies_silver  |381109|policies_silver|2025-11-28 10:36:03.700764|
|distinct_policy_ids       |381109|policies_silver|2025-11-25 01:31:29.412102|
|rowcount_policies_silver  |381109|policies_silver|2025-11-25 01:31:06.610274|
+--------------------------+------+---------------+--------------------------+



# 🔹 Cell 10 — Final sanity check & sample

In [17]:
# --- Cell 10: Final sanity & sample view ---

print("Silver Members row count:", members_silver.count())
print("Distinct Member_IDs:", members_silver.select("Member_ID").distinct().count())
print("Distinct Policy_IDs:", members_silver.select("Policy_ID").distinct().count())

print("\nSample Silver Members:")
members_silver.show(10, truncate=False)


Silver Members row count: 381109


Distinct Member_IDs: 381109


Distinct Policy_IDs: 381109

Sample Silver Members:


+------------+----------+---+------+------------------+------+---------------+-----------------+------+------------+------------+--------+------------+------------+
|Member_ID   |Policy_ID |Age|Gender|BMI               |Smoker|Chronic_Disease|Employment_Status|Region|dq_age_valid|dq_bmi_valid|Age_Band|BMI_Category|Risk_Segment|
+------------+----------+---+------+------------------+------+---------------+-----------------+------+------------+------------+--------+------------+------------+
|MEM_00000008|P_00000008|56 |F     |23.140940890221643|Y     |Asthma         |Student          |280   |1           |1           |41-60   |Normal      |High        |
|MEM_00001714|P_00001714|71 |M     |27.963719840043765|N     |None           |Employed         |360   |1           |1           |60+     |Overweight  |Medium      |
|MEM_00001746|P_00001746|75 |M     |30.541100957674118|N     |Diabetes       |Student          |80    |1           |1           |60+     |Obese       |High        |
|MEM_00001

# 🔷 Cell A — Bronze vs Silver column comparison (Members)

In [18]:
# --- Cell A: Bronze vs Silver column comparison (Members) ---

bronze_members = spark.read.format("delta").load(BRONZE_MEMBERS_PATH)
silver_members = spark.read.format("delta").load(SILVER_MEMBERS_PATH)

bronze_cols = set(bronze_members.columns)
silver_cols = set(silver_members.columns)

new_features     = sorted(list(silver_cols - bronze_cols))
dropped_features = sorted(list(bronze_cols - silver_cols))
common_features  = sorted(list(bronze_cols & silver_cols))

print("A1) Column comparison:")
print("  New in Silver:   ", new_features)
print("  Dropped in Silver:", dropped_features)
print("  Common columns:  ", len(common_features))


A1) Column comparison:
  New in Silver:    ['Age_Band', 'BMI_Category', 'Risk_Segment', 'dq_age_valid', 'dq_bmi_valid']
  Dropped in Silver: []
  Common columns:   9


# 🔷 Cell B — Per-column type & value changes (Members)

In [19]:
# --- Cell B: Type & value changes per common column (Members) ---

bronze_types = dict(bronze_members.dtypes)
silver_types = dict(silver_members.dtypes)

# Align by Member_ID for a like-for-like comparison
br_aligned = bronze_members.select("Member_ID", *common_features).dropDuplicates(["Member_ID"])
sv_aligned = silver_members.select("Member_ID", *common_features).dropDuplicates(["Member_ID"])

joined = (
    br_aligned.alias("bz")
    .join(sv_aligned.alias("sv"), on="Member_ID", how="outer")
)

for col in common_features:
    bronze_type = bronze_types.get(col)
    silver_type = silver_types.get(col)
    type_changed = bronze_type != silver_type
    type_msg = (
        f"Type changed: {bronze_type} → {silver_type}"
        if type_changed else
        f"Type unchanged: {bronze_type}"
    )

    n_diff = (
        joined
        .filter(
            (F.col(f"bz.{col}").isNull() & F.col(f"sv.{col}").isNotNull()) |
            (F.col(f"bz.{col}").isNotNull() & F.col(f"sv.{col}").isNull()) |
            (F.col(f"bz.{col}") != F.col(f"sv.{col}"))
        )
        .count()
    )

    if n_diff == 0 and not type_changed:
        print(f"  • '{col}' unchanged. {type_msg}")
    else:
        print(f"  • '{col}' transformed in {n_diff} members. {type_msg}")
        sample_diff = (
            joined
            .filter(
                (F.col(f"bz.{col}").isNull() & F.col(f"sv.{col}").isNotNull()) |
                (F.col(f"bz.{col}").isNotNull() & F.col(f"sv.{col}").isNull()) |
                (F.col(f"bz.{col}") != F.col(f"sv.{col}"))
            )
            .select("Member_ID", F.col(f"bz.{col}").alias("bronze"), F.col(f"sv.{col}").alias("silver"))
            .limit(5)
        )
        print(f"    Sample differences for '{col}':")
        sample_diff.show(truncate=False)


  • 'Age' unchanged. Type unchanged: int


  • 'BMI' unchanged. Type unchanged: double


  • 'Chronic_Disease' unchanged. Type unchanged: string


  • 'Employment_Status' transformed in 14115 members. Type unchanged: string
    Sample differences for 'Employment_Status':


+------------+-------------+-------------+
|Member_ID   |bronze       |silver       |
+------------+-------------+-------------+
|MEM_00000096|Self-Employed|Self-employed|
|MEM_00000219|Self-Employed|Self-employed|
|MEM_00000239|Self-Employed|Self-employed|
|MEM_00000355|Self-Employed|Self-employed|
|MEM_00000520|Self-Employed|Self-employed|
+------------+-------------+-------------+



...

+------------+------+------+
|Member_ID   |bronze|silver|
+------------+------+------+
|MEM_00000008|Female|F     |
|MEM_00000009|Female|F     |
|MEM_00000018|Female|F     |
|MEM_00000023|Male  |M     |
|MEM_00000024|Male  |M     |
+------------+------+------+



  • 'Member_ID' unchanged. Type unchanged: string


  • 'Policy_ID' unchanged. Type unchanged: string


  • 'Region' unchanged. Type unchanged: string


  • 'Smoker' unchanged. Type unchanged: string


# 🔷 Cell C — Explain new Silver-only features (Members)

In [20]:
# --- Cell C: New Silver-only features & business purpose (Members) ---

print("\nC1) New Silver-only features (Members):", new_features)

for col in new_features:
    print(f"\n  ➕ '{col}' (business purpose):")
    if col == "dq_age_valid":
        print("     → Data quality flag: Age within expected range [0,110] or null.")
    elif col == "dq_bmi_valid":
        print("     → Data quality flag: BMI within safe range [10,60] or null.")
    elif col == "Age_Band":
        print("     → Segments members into age bands for cohort analysis and pricing decisions.")
    elif col == "BMI_Category":
        print("     → Classifies members into Underweight/Normal/Overweight/Obese for health-risk modeling.")
    elif col == "Risk_Segment":
        print("     → Combines Age, BMI, Smoker status and Chronic_Disease into a simple risk band (Low/Medium/High).")
    else:
        print("     → Derived Silver feature; see transformation logic for details.")

    print(f"     Sample data for '{col}':")
    silver_members.select("Member_ID", col).limit(5).show(truncate=False)



C1) New Silver-only features (Members): ['Age_Band', 'BMI_Category', 'Risk_Segment', 'dq_age_valid', 'dq_bmi_valid']

  ➕ 'Age_Band' (business purpose):
     → Segments members into age bands for cohort analysis and pricing decisions.
     Sample data for 'Age_Band':


+------------+--------+
|Member_ID   |Age_Band|
+------------+--------+
|MEM_00000008|41-60   |
|MEM_00001714|60+     |
|MEM_00001746|60+     |
|MEM_00001967|25-40   |
|MEM_00001995|60+     |
+------------+--------+


  ➕ 'BMI_Category' (business purpose):
     → Classifies members into Underweight/Normal/Overweight/Obese for health-risk modeling.
     Sample data for 'BMI_Category':


+------------+------------+
|Member_ID   |BMI_Category|
+------------+------------+
|MEM_00000008|Normal      |
|MEM_00001714|Overweight  |
|MEM_00001746|Obese       |
|MEM_00001967|Overweight  |
|MEM_00001995|Obese       |
+------------+------------+


  ➕ 'Risk_Segment' (business purpose):
     → Combines Age, BMI, Smoker status and Chronic_Disease into a simple risk band (Low/Medium/High).
     Sample data for 'Risk_Segment':


+------------+------------+
|Member_ID   |Risk_Segment|
+------------+------------+
|MEM_00000008|High        |
|MEM_00001714|Medium      |
|MEM_00001746|High        |
|MEM_00001967|High        |
|MEM_00001995|High        |
+------------+------------+


  ➕ 'dq_age_valid' (business purpose):
     → Data quality flag: Age within expected range [0,110] or null.
     Sample data for 'dq_age_valid':


+------------+------------+
|Member_ID   |dq_age_valid|
+------------+------------+
|MEM_00000008|1           |
|MEM_00001714|1           |
|MEM_00001746|1           |
|MEM_00001967|1           |
|MEM_00001995|1           |
+------------+------------+


  ➕ 'dq_bmi_valid' (business purpose):
     → Data quality flag: BMI within safe range [10,60] or null.
     Sample data for 'dq_bmi_valid':


+------------+------------+
|Member_ID   |dq_bmi_valid|
+------------+------------+
|MEM_00000008|1           |
|MEM_00001714|1           |
|MEM_00001746|1           |
|MEM_00001967|1           |
|MEM_00001995|1           |
+------------+------------+



# 🔷 Cell D — Snapshot metrics & DQ improvements (Members)

In [21]:
# --- Cell D: Snapshot & DQ improvements for Members ---

from pyspark.sql import functions as F

def pct(x, n):
    return 0.0 if n == 0 else round(100.0 * x / n, 4)

br = bronze_members
sv = silver_members

n_br = br.count()
n_sv = sv.count()

print("D1) Snapshot row counts:")
print({
    "rows_bronze_members": n_br,
    "rows_silver_members": n_sv,
    "distinct_member_id_bronze": br.select("Member_ID").distinct().count(),
    "distinct_member_id_silver": sv.select("Member_ID").distinct().count(),
})

# DQ issues in Bronze
br_null_keys = br.filter(F.col("Member_ID").isNull() | F.col("Policy_ID").isNull()).count()
br_bad_age   = br.filter(~(F.col("Age").isNull() | ((F.col("Age") >= 0) & (F.col("Age") <= 110)))).count()
br_bad_bmi   = br.filter(~(F.col("BMI").isNull() | ((F.col("BMI") >= 10) & (F.col("BMI") <= 60)))).count()

# DQ issues in Silver (use dq flags)
sv_null_keys = sv.filter(F.col("Member_ID").isNull() | F.col("Policy_ID").isNull()).count()
sv_bad_age   = sv.filter(F.col("dq_age_valid") == 0).count()
sv_bad_bmi   = sv.filter(F.col("dq_bmi_valid") == 0).count()

report = {
    "bronze_null_keys_pct": pct(br_null_keys, n_br),
    "silver_null_keys_pct": pct(sv_null_keys, n_sv),
    "bronze_bad_age_pct":   pct(br_bad_age, n_br),
    "silver_bad_age_pct":   pct(sv_bad_age, n_sv),
    "bronze_bad_bmi_pct":   pct(br_bad_bmi, n_br),
    "silver_bad_bmi_pct":   pct(sv_bad_bmi, n_sv),
}

print("\nD2) DQ improvement report (Members):")
print(report)


D1) Snapshot row counts:


{'rows_bronze_members': 381109, 'rows_silver_members': 381109, 'distinct_member_id_bronze': 381109, 'distinct_member_id_silver': 381109}

D2) DQ improvement report (Members):
{'bronze_null_keys_pct': 0.0, 'silver_null_keys_pct': 0.0, 'bronze_bad_age_pct': 0.0, 'silver_bad_age_pct': 0.0, 'bronze_bad_bmi_pct': 0.0, 'silver_bad_bmi_pct': 0.0}


25/11/29 02:10:33 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 343961 ms exceeds timeout 120000 ms
25/11/29 02:10:33 WARN SparkContext: Killing executors is not supported by current scheduler.
25/11/29 02:10:41 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$